# Задание - Оптимизация инференса

## 1. Xарактеристики локального устройства

### CPU
1. Модель: 12th Gen Intel(R) Core(TM) i7-12700H
2. Физические ядра: 14
3. Производительные ядра: 6
4. Энергоэффективные ядра: 8
5. Базовая тактовая частота: 2,3 ГГц
6. Максимальная частота: 4,7 ГГц

### GPU
1. Модель: NVIDIA GeForce RTX 3060
2. Объем видеопамяти: 6 GB

### DRAM
1. Общий объем: 16 GB

## 2. Выбор модели qwen3.5

С учетом ограничения GPU возьмем Qwen3.5-4B_Q8_0 - модель на 4B параметров с 8bit квантованием.
Такая модель займет 1.2 * 4  * 10^9 * 1 = 4.47 GB, обеспечивая запас в 1.5 GB (до 32 768 токенов в kv-кеше)

# 3. Запуск llama.cpp

In [57]:
import subprocess
import time

Определим путь до модели, хост, порт и базовые аргументы

In [58]:
PATH = "./Qwen3.5-4B-Q8_0.gguf"
PORT = "8080"
HOST = "127.0.0.1"

In [59]:
base_args = [("-m", PATH), ("--host", HOST), ("--port", PORT)]

Далее будем для каждого эксперимента выбирать и изменять 1 из 4 настроек, добавляя к базовым:
1. -ngl - сколько слоев разместить на GPU
2. -b - реальный размер батча
3. -ub - логический микробатч (на какие куски будет разделен реальный батч)
4. --flash-attn - использование/не использование Flash Attention

In [60]:
# base_args += [("-ngl", "0")]
# base_args += [("-ngl", "12")]
# base_args += [("-ngl", "all")]

# base_args += [("-b", "512")]
# base_args += [("-b", "4")]
# base_args += [("-b", "2")]
# base_args += [("-b", "64")]
# base_args += [("-b", "128")]
# base_args += [("-b", "4096")]

# base_args += [("--flash-attn", "off")]
# base_args += [("--flash-attn", "on")]

# base_args += [("-ub", "2048")]
# base_args += [("-ub", "2")]
# base_args += [("-ub", "32")]
# base_args += [("-ub", "128")]

base_args += [("-b", "128"), ("-ub", "32"), ("--flash-attn", "on"), ("-ngl", "all")]

In [61]:
args_array = ["llama-server"]

for k, v in base_args:
    args_array += [k, v]

args_array

['llama-server',
 '-m',
 './Qwen3.5-4B-Q8_0.gguf',
 '--host',
 '127.0.0.1',
 '--port',
 '8080',
 '-b',
 '128',
 '-ub',
 '32',
 '--flash-attn',
 'on',
 '-ngl',
 'all']

In [62]:
process = subprocess.Popen(args_array)

Пауза для подготовки модели:

In [63]:
time.sleep(15)

# 4. Выполнение замеров

In [64]:
import requests
import time
import numpy as np
import os
from tqdm import tqdm

Проведем n=3 замеров для более стабильного результата на фиксированном запросе

In [65]:
URL = f"http://{HOST}:{PORT}/v1/chat/completions"
RUNS = 3
REPORTS = "./reports/"
TEST_PAYLOAD = {
    "model": "local",
    "messages": [
        {"role": "system", "content": "You are a helpful AI-assistnat."},
        {"role": "user", "content": "Write a Python function to calculate factorial recursively"}
    ],
    "max_tokens": 8192,
    "temperature": 0,
    "stream": True
}

In [66]:
def mesuare(payload):
    start = time.perf_counter()

    response = requests.post(URL, json=payload, stream=True)

    first_token_time = None
    tokens = 0

    for line in tqdm(response.iter_lines()):

        if line:

            if first_token_time is None: # встречаем первый токен
                first_token_time = time.perf_counter()

            tokens += 1

    end = time.perf_counter()

    ttft = first_token_time - start
    total = end - start

    tpot = (total - ttft) / max(tokens - 1, 1)

    return ttft * 1000, tpot * 1000

In [67]:
next_report = len(os.listdir(REPORTS)) + 1
save_path = os.path.join(REPORTS, f"res-{next_report}.txt")

ttfts = []
tpots = []

ttft, tpot = mesuare(TEST_PAYLOAD) #для разогрева, поскольку для первого запроса метрки будут выше

for i in range(RUNS):

    ttft, tpot = mesuare(TEST_PAYLOAD)

    ttfts.append(ttft)
    tpots.append(tpot)

    print(f"Run {i+1}")
    print(f"TTFT: {ttft:.2f} ms")
    print(f"TPOT: {tpot:.2f} ms/token")
    print()

986it [00:08, 110.41it/s]
986it [00:08, 109.65it/s]


Run 1
TTFT: 94.48 ms
TPOT: 18.28 ms/token



986it [00:08, 109.57it/s]


Run 2
TTFT: 74.88 ms
TPOT: 18.29 ms/token



986it [00:08, 109.57it/s]

Run 3
TTFT: 76.20 ms
TPOT: 18.29 ms/token



In [68]:
print(f"Results")
print(f"TTFT: {np.mean(ttfts):.2f} +- {np.std(ttfts):.2f} ms")
print(f"TPOT: {np.mean(tpots):.2f} +- {np.std(tpots):.2f} ms/token")
print()

Results
TTFT: 81.85 +- 8.95 ms
TPOT: 18.29 +- 0.01 ms/token



In [69]:
if (not os.path.exists(save_path)):
    with open(save_path, "w", encoding="utf-8") as af:
        af.write(f"id\tttft_mean\tttft_std\ttpot_mean\ttpot_std\n")
        for i, zp in enumerate(zip(ttfts, tpots)):
            ttft, tpot = zp
            output_str = f"{i}\t{ttft}\t0\t{tpot}\t0\n"
            af.write(output_str)

        af.write(f"0\t{np.mean(ttfts)}\t{np.std(ttfts)}\t{np.mean(tpots)}\t{np.std(tpots)}\n")
else:
    print("Файл уже существует!")

In [70]:
process.terminate()

## 5. Результаты

### Базовый замер
| TTFT (mean) | TTFT (std)  | TPOF (mean) | TPOF (std) |
|-------------|-------------|-------------|------------|
| 590.35      | 22.02       | 62.82       | 0.75       |

Параметры по умолчанию в llama.cpp:
* -ngl = "auto"
* -b = 2048
* --flash-attn = "auto"
* -ub = 512

### Влияние параметров
| параметр     | Значение | TTFT (mean) | TTFT (std) | TPOF (mean) | TPOF (std) |
|--------------|----------|-------------|------------|-------------|------------|
| -ngl         | 0        | 1426.50     | 54.30      | 154.11      | 1.27       |
| -ngl         | 12       | 1046.93     | 11.27      | 116.73      | 0.30       |
| -ngl         | "all"    | 82.14       | 11.72      | 18.33       | 0.01       |
| -b           | 2        | 1204.27     | 30.75      | 39.61       | 0.22       |        
| -b           | 4        | 691.30      | 7.50       | 39.66       | 0.24       |    
| -b           | 64       | 411.17      | 10.75      | 45.00       | 0.37       |
| -b           | 128      | 419.72      | 18.19      | 44.46       | 0.30       |
| -b           | 512      | 583.06      | 8.50       | 63.80       | 1.48       | 
| -b           | 4096     | 589.39      | 16.44      | 66.63       | 0.35       | 
| --flash-attn | "off"    | 676.88      | 150.94     | 64.66       | 1.62       |
| --flash-attn | "on"     | 595.32      | 15.66      | 65.73       | 0.75       |
| -ub          | 2        | 1124.05     | 13.37      | 37.84       | 0.14       |        
| -ub          | 32       | 435.78      | 17.05      | 44.74       | 0.72       |    
| -ub          | 128      | 414.26      | 14.90      | 45.05       | 0.71       |
| -ub          | 2048     | 1153.00     | 24.98      | 119.37      | 2.33       |

Из таблицы следует, что:
1. Увеличение -ngl (количества слоев, размещенных на GPU) значительно уменьшает TTFT и TPOF за счет переноса вычислений с CPU на GPU. Полное размещение модели на GPU дало наилучшие результаты.
2. Измение -b и -ub имеет нелинейный характер для TTFT: по мере увеличения -b и -ub TTFT сначала падает, а потом начинает подрастать. TPOT при увеличении значения растет, но имеет плато в районе 32-128
3. Flash Attention при "on" уменьшило TTFT и сделало вывод стабильнее (уменьшение std)

### Итоговый замер
Для итогово замера выберем следующие параметры:
* -ngl = "all"
* -b = 128
* --flash-attn = "on"
* -ub = 32

| TTFT (mean) | TTFT (std)  | TPOF (mean) | TPOF (std) |
|-------------|-------------|-------------|------------|
| 81.97       | 6.25        | 18.31       | 0.01       |

Таким образом, TTFT снизилось - в 7-8 раз, а TPOF снизилось в 4 раза. При этом загруженность GPU выросла с 3.9GB до 4.6GB